In [ ]:
%pip install lightgbm catboost scikit-learn pandas numpy matplotlib

In [ ]:
import pandas as pd
import zipfile
import os
import numpy as np

print("="*80)
print("ECOGRID AI: DATASET INTEGRATION PIPELINE (OPTIMIZED FEATURE MATRIX)")
print("="*80)

# 1. Load Local UCI Occupancy Dataset
print("\n[STEP 1/3] Loading UCI Occupancy Detection dataset...")
second_dataset_path = "occupancy+detection.zip"
extract_dir = "occupancy_extracted"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(second_dataset_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

occupancy_train = pd.read_csv(os.path.join(extract_dir, 'datatraining.txt'))
occupancy_test1 = pd.read_csv(os.path.join(extract_dir, 'datatest.txt'))
occupancy_test2 = pd.read_csv(os.path.join(extract_dir, 'datatest2.txt'))

occupancy_combined = pd.concat([occupancy_train, occupancy_test1, occupancy_test2], ignore_index=True)
print(f"  - Combined occupancy raw records: {occupancy_combined.shape}")

# 2. Feature Engineering
print("\n[STEP 2/3] Extracting Signal Features from Occupancy Data...")
occupancy_combined['date'] = pd.to_datetime(occupancy_combined['date'])
occupancy_combined['hourly_timestamp'] = occupancy_combined['date'].dt.floor('h')

occupancy_features = occupancy_combined.groupby('hourly_timestamp').agg({
    'Temperature': 'mean',
    'Humidity': 'mean',
    'Light': 'mean',
    'CO2': 'mean',
    'HumidityRatio': 'mean',
    'Occupancy': 'max'
}).reset_index()

occupancy_features = occupancy_features.rename(columns={
    'Temperature': 'Ambient_Temp_C',
    'CO2': 'CO2_Level',
    'Occupancy': 'Occupancy_State_Binary'
})

occupancy_features['Hour'] = occupancy_features['hourly_timestamp'].dt.hour
occupancy_features['DayOfWeek'] = occupancy_features['hourly_timestamp'].dt.dayofweek
occupancy_features['IsWeekend'] = (occupancy_features['DayOfWeek'] >= 5).astype(int)

# Cyclical time transformation & Rolling Features
occupancy_features['Hour_Sin'] = np.sin(2 * np.pi * occupancy_features['Hour'] / 24.0)
occupancy_features['Hour_Cos'] = np.cos(2 * np.pi * occupancy_features['Hour'] / 24.0)
occupancy_features['Temp_Rolling_Mean'] = occupancy_features['Ambient_Temp_C'].rolling(window=3, min_periods=1).mean()
occupancy_features['CO2_Rolling_Mean'] = occupancy_features['CO2_Level'].rolling(window=3, min_periods=1).mean()
occupancy_features['Light_Rolling_Mean'] = occupancy_features['Light'].rolling(window=3, min_periods=1).mean()

# Corrected Categorical Mapping Logic
def map_occupancy_category(row):
    if row['Occupancy_State_Binary'] == 0:
        return 'Low'
    else:
        if row['CO2_Level'] > 850 or row['Light'] > 200:
            return 'High'
        return 'Medium'

occupancy_features['Occupancy_Category'] = occupancy_features.apply(map_occupancy_category, axis=1)

# Thermodynamic Load Modeling
occupancy_weights = {"High": 28.5, "Medium": 14.0, "Low": 3.5}
np.random.seed(42)
occupancy_features['HVAC_Power_kW'] = occupancy_features.apply(
    lambda r: round(12.0 + occupancy_weights[r["Occupancy_Category"]] + 
                   max(0, (r['Ambient_Temp_C'] - 20) * 1.8) + 
                   np.random.normal(0, 0.5), 2),
    axis=1
)

final_features = occupancy_features[[
    'hourly_timestamp',
    'Hour_Sin', 
    'Hour_Cos', 
    'DayOfWeek', 
    'IsWeekend', 
    'Ambient_Temp_C',
    'Temp_Rolling_Mean',
    'CO2_Level',
    'CO2_Rolling_Mean',
    'Light',
    'Light_Rolling_Mean',
    'Humidity',
    'Occupancy_Category',
    'HVAC_Power_kW'
]].copy()

final_features.to_csv('ecogrid_integrated_matrix.csv', index=False)
print("="*80)
print("✅ ECOGRID FEATURE MATRIX SAVED: ecogrid_integrated_matrix.csv")
print("="*80)

In [ ]:
# =====================================================================
# ECOGRID AI: MULTI-TASK HETEROGENEOUS GRADIENT BOOSTING ENSEMBLE
# Production Build | Fully Featurized & Optimal Tuning
# =====================================================================

import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

def console_log(msg: str, delay: float = 0.01):
    """Outputs structured log messages directly to standard output."""
    print(msg, flush=True)
    time.sleep(delay)

class EcoGridEngine:
    def __init__(self):
        self.label_encoder = LabelEncoder()
        
        # Complete Feature Set (Temporal + Environmental Sensors)
        self.feature_cols = [
            'Hour_Sin', 'Hour_Cos', 'DayOfWeek', 'IsWeekend', 
            'Ambient_Temp_C', 'Temp_Rolling_Mean', 
            'CO2_Level', 'CO2_Rolling_Mean', 
            'Light', 'Light_Rolling_Mean', 'Humidity'
        ]

        # Hyperparameter Tuning for Optimal Bias-Variance Balance
        self.lgb_params_cls = {
            'n_estimators': 100, 
            'max_depth': 5, 
            'num_leaves': 15,
            'learning_rate': 0.05, 
            'reg_lambda': 1.0, 
            'random_state': 42, 
            'verbose': -1
        }
        self.cat_params_cls = {
            'iterations': 100, 
            'depth': 5, 
            'learning_rate': 0.05, 
            'l2_leaf_reg': 2.0, 
            'random_state': 42, 
            'verbose': 0
        }
        
        self.lgb_params_reg = {
            'n_estimators': 100, 
            'max_depth': 4, 
            'num_leaves': 10,
            'learning_rate': 0.05, 
            'reg_lambda': 1.0, 
            'random_state': 42, 
            'verbose': -1
        }
        self.cat_params_reg = {
            'iterations': 100, 
            'depth': 4, 
            'learning_rate': 0.05, 
            'l2_leaf_reg': 2.0, 
            'random_state': 42, 
            'verbose': 0
        }

    def run_pipeline(self):
        console_log("="*80)
        console_log(" 🚀 ECOGRID AI: MULTI-TASK GRADIENT BOOSTING PIPELINE")
        console_log("="*80)

        # 1. Load Dataset
        console_log("\n[STEP 1/4] Loading Integrated EcoGrid Dataset...")
        try:
            df = pd.read_csv('ecogrid_integrated_matrix.csv')
            console_log(f" ↳ Loaded integrated dataset: {df.shape}")
        except FileNotFoundError:
            console_log(" ❌ Error: ecogrid_integrated_matrix.csv not found.")
            return

        # 2. Data Preprocessing
        console_log("\n[STEP 2/4] Preprocessing Data...")
        df = df.dropna(subset=['HVAC_Power_kW', 'Occupancy_Category'])
        df['Occupancy_Label'] = self.label_encoder.fit_transform(df['Occupancy_Category'])
        
        X = df[self.feature_cols]
        y_cls = df["Occupancy_Label"]
        y_reg = df["HVAC_Power_kW"]

        # Stratified Train/Test Partitioning
        X_train, X_test, y_train_cls, y_test_cls, y_train_reg, y_test_reg = train_test_split(
            X, y_cls, y_reg, test_size=0.2, random_state=42, stratify=y_cls
        )

        console_log(f" ↳ Dataset: {len(df)} Rows | Train Set: {len(X_train)} Samples | Test Set: {len(X_test)} Samples")

        # 3. Model Training
        console_log("\n[STEP 3/4] Fitting Heterogeneous Gradient Boosting Ensemble...")
        m1_cls = lgb.LGBMClassifier(**self.lgb_params_cls).fit(X_train, y_train_cls)
        m2_cls = CatBoostClassifier(**self.cat_params_cls).fit(X_train, y_train_cls)

        m1_reg = lgb.LGBMRegressor(**self.lgb_params_reg).fit(X_train, y_train_reg)
        m2_reg = CatBoostRegressor(**self.cat_params_reg).fit(X_train, y_train_reg)

        # 4. Metric Extraction
        console_log("\n[STEP 4/4] Evaluating Model Generalization Metrics...")
        tr_prob = (m1_cls.predict_proba(X_train) + m2_cls.predict_proba(X_train)) / 2
        te_prob = (m1_cls.predict_proba(X_test) + m2_cls.predict_proba(X_test)) / 2
        train_acc = accuracy_score(y_train_cls, np.argmax(tr_prob, axis=1))
        test_acc = accuracy_score(y_test_cls, np.argmax(te_prob, axis=1))

        te_pred_reg = (m1_reg.predict(X_test) + m2_reg.predict(X_test)) / 2
        tr_pred_reg = (m1_reg.predict(X_train) + m2_reg.predict(X_train)) / 2
        train_rmse = np.sqrt(mean_squared_error(y_train_reg, tr_pred_reg))
        test_rmse = np.sqrt(mean_squared_error(y_test_reg, te_pred_reg))

        console_log(f" 📑 Classification Accuracy -> Train: {train_acc*100:.1f}% | Test: {test_acc*100:.1f}%")
        console_log(f" 📑 Forecasting RMSE       -> Train: {train_rmse:.2f} kW | Test: {test_rmse:.2f} kW")

        # Live API Emulation
        self.live_api_endpoint(
            m1_cls, m2_cls, m1_reg, m2_reg, 
            hour=14, day_of_week=2, temp=34.5, temp_avg=33.8, 
            co2=950.0, co2_avg=910.0, light=450.0, light_avg=420.0, humidity=28.5
        )

        # Output Chart Plot
        self.plot_and_save(df, y_test_reg, te_pred_reg)

    def live_api_endpoint(self, m1_cls, m2_cls, m1_reg, m2_reg, hour, day_of_week, temp, temp_avg, co2, co2_avg, light, light_avg, humidity):
        console_log("\n[STEP 5/5] Executing Live Production API Emulation...")
        hour_sin = np.sin(2 * np.pi * hour / 24.0)
        hour_cos = np.cos(2 * np.pi * hour / 24.0)
        is_weekend = 1 if day_of_week >= 5 else 0

        payload = pd.DataFrame([[
            hour_sin, hour_cos, day_of_week, is_weekend, 
            temp, temp_avg, co2, co2_avg, light, light_avg, humidity
        ]], columns=self.feature_cols)

        voted_probs = (m1_cls.predict_proba(payload) + m2_cls.predict_proba(payload)) / 2
        assigned_state = self.label_encoder.classes_[np.argmax(voted_probs, axis=1)[0]]

        blended_pred = (m1_reg.predict(payload)[0] + m2_reg.predict(payload)[0]) / 2

        console_log(f" ↳ Incoming Telemetry --> {hour}:00 | Temp: {temp}°C | CO2: {co2}ppm | Light: {light}lx")
        console_log(f" ↳ Spatial Model State --> [{assigned_state.upper()}] Occupancy")
        console_log(f" ↳ Kinetic Forecast    --> [{blended_pred:.2f} kW] Electrical Load")

        if assigned_state == "High" and blended_pred > 35.0:
            console_log(" 🚨 ROUTING ACTION    --> PRE-COOLING ENGAGED. Buffering peak spike.")
        elif assigned_state == "Low":
            console_log(" 🍃 ROUTING ACTION    --> DEEP HIBERNATION MODE ENGAGED.")
        else:
            console_log(" ⚖️ ROUTING ACTION    --> STEADY STATE MAINTAINED.")
        console_log("="*80)

    def plot_and_save(self, df, y_test_reg, te_pred_reg):
        plt.figure(figsize=(12, 5))
        plt.plot(y_test_reg.values, label="Actual Load (kW)", color="steelblue", linewidth=2)
        plt.plot(te_pred_reg, label="Predicted Load (kW)", color="darkorange", linewidth=2, linestyle="--")
        plt.title("EcoGrid AI: Stratified Test Set - Actual vs Predicted Load", fontsize=12, fontweight="bold")
        plt.xlabel("Test Sample Index", fontsize=10)
        plt.ylabel("Power (kW)", fontsize=10)
        plt.legend(fontsize=10)
        plt.tight_layout()
        plt.savefig("ecogrid_real_data_predictions.png", dpi=150)
        console_log(" 📊 Prediction plot saved: ecogrid_real_data_predictions.png")

if __name__ == "__main__":
    engine = EcoGridEngine()
    engine.run_pipeline()